# ❤️ التنبؤ بمخاطر أمراض القلب (Heart Disease Risk Classifier)
### مشروع B5 — مسار تعلم الآلة (Machine Learning Track)

---
## 🎯 المشكلة التجارية (Business Problem)
عيادة عايزة **أداة فرز (Screening)** تتنبأ باحتمال إصابة المريض بمرض قلب من فحوصاته الأساسية،
عشان توجّه الأطباء للحالات الخطيرة بدري. **قرار طبي حسّاس → التفسير (Interpretability) مهم جداً.**

**نوع المشكلة:** تصنيف ثنائي (مريض / سليم).

## 📦 ما الذي يثبته المشروع
كشف **تسريب البيانات (Data Leakage)** من ميزة مشتقة · Pipeline · مقارنة موديلات ·
تقييم (ROC-AUC) · **تفسير العوامل الطبية (Feature Importance)**.


## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

In [ ]:
# 🚀 إعداد تلقائي — Colab / Kaggle / محلي (no-op محلياً)
import os, sys, subprocess, importlib.util
REPO    = "https://github.com/ahmedelgendy11/AI-and-data-science-portfolio"
PROJECT = "ml/b5_heart_disease"          # مسار المشروع داخل portfolio/
PKGS    = ["xgboost"]          # مكتبات المشروع (تتثبّت لو ناقصة)
for _pkg in PKGS:
    if importlib.util.find_spec(_pkg.replace("-", "_")) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg])
if not os.path.isdir("data"):                 # سحابياً: نجيب الريبو ونروح لمجلد المشروع
    _clone = REPO.rstrip("/").split("/")[-1]
    if not os.path.isdir(_clone):
        subprocess.run(["git", "clone", "-q", REPO + ".git"])
    os.chdir(os.path.join(_clone, "portfolio", PROJECT))
print("✓ جاهز —", os.getcwd())

## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

## 📚 قبل ما تبدأ — محتاج تذاكر إيه
| المفهوم | المصدر | بيُستخدم في إيه |
|---|---|---|
| التصنيف (LogReg, RF) | ISLR (ch.4) / Géron | أساس التنبؤ الثنائي |
| **Data Leakage من ميزة مشتقة** | Huyen / Géron | ميزة "مثالية" غالباً غش — لازم تكتشفها |
| ROC-AUC + Confusion Matrix | ISLR (ch.4) | تقييم مع أهمية الحالات الإيجابية |
| Feature Importance | Géron / Molnar | في الطب لازم تعرف "ليه" |

> 🎯 **بيُستخدم في الواقع:** أنظمة الدعم الطبي، فرز المرضى، التأمين الصحي، الطب الوقائي.
> ⚠️ **تنبيه أخلاقي:** أداة فرز مساعِدة فقط — مش بديل عن تشخيص الطبيب.


## 0️⃣ المكتبات

In [1]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_style('whitegrid'); np.random.seed(42)
print('ready ✓')

ready ✓


## 1️⃣ تحميل واستكشاف البيانات (EDA)

In [2]:
df = pd.read_csv('data/health_risk_data.csv')
print('Shape:', df.shape, '| Disease rate: {:.1%}'.format(df['heart_disease'].mean()))
corr = df.corr(numeric_only=True)['heart_disease'].sort_values(ascending=False)
print(corr.round(2))

Shape: (2000, 13) | Disease rate: 45.2%
heart_disease              1.00
risk_score                 0.80
age                        0.63
cholesterol                0.59
systolic_bp                0.53
diastolic_bp               0.37
bmi                        0.27
blood_sugar                0.27
exercise_hours_per_week    0.01
Name: heart_disease, dtype: float64


## 2️⃣ ⚠️ كشف تسريب البيانات (Data Leakage)
لاحظ إن `risk_score` ارتباطه بالهدف **عالي جداً (~0.80)**. ده مش ميزة حقيقية — ده **درجة خطورة محسوبة
من النتيجة نفسها**. لو سِبناها، الموديل هيغش وهيفشل في الواقع. **نشيلها.**

In [3]:
LEAKY = ['risk_score']
X = df.drop(columns=['heart_disease', 'patient_id'] + LEAKY)
y = df['heart_disease']
print('Dropped leaky feature:', LEAKY)
print('Features used:', list(X.columns))

Dropped leaky feature: ['risk_score']
Features used: ['age', 'gender', 'bmi', 'systolic_bp', 'diastolic_bp', 'cholesterol', 'blood_sugar', 'smoking_status', 'exercise_hours_per_week', 'family_history']


## 3️⃣ المعالجة والتقسيم (Pipeline)

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
num = X.select_dtypes('number').columns.tolist()
cat = X.select_dtypes('object').columns.tolist()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat)])
print('num:', num, '| cat:', cat)

num: ['age', 'bmi', 'systolic_bp', 'diastolic_bp', 'cholesterol', 'blood_sugar', 'exercise_hours_per_week'] | cat: ['gender', 'smoking_status', 'family_history']


## 4️⃣ مقارنة الموديلات (Cross-Validation, ROC-AUC)

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
cv = StratifiedKFold(5, shuffle=True, random_state=42)
models = {
    'LogReg': LogisticRegression(max_iter=1000),
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                             eval_metric='logloss', random_state=42)}
for n, m in models.items():
    auc = cross_val_score(Pipeline([('p',pre),('m',m)]), X_tr, y_tr, cv=cv, scoring='roc_auc').mean()
    print(f'{n:14} ROC-AUC = {auc:.3f}')

LogReg         ROC-AUC = 0.906


RandomForest   ROC-AUC = 0.898


XGBoost        ROC-AUC = 0.888


## 5️⃣ التقييم النهائي

In [6]:
from sklearn.metrics import classification_report, confusion_matrix, RocCurveDisplay
best = Pipeline([('p',pre),('m',models['LogReg'])]).fit(X_tr, y_tr)
proba = best.predict_proba(X_te)[:,1]; pred = (proba>=0.5).astype(int)
print(classification_report(y_te, pred, target_names=['Healthy','Disease']))
fig, ax = plt.subplots(1,2, figsize=(12,4))
sns.heatmap(confusion_matrix(y_te, pred), annot=True, fmt='d', cmap='Reds', ax=ax[0],
            xticklabels=['Healthy','Disease'], yticklabels=['Healthy','Disease'])
ax[0].set_title('Confusion Matrix')
RocCurveDisplay.from_predictions(y_te, proba, ax=ax[1]); ax[1].set_title('ROC Curve')
plt.tight_layout(); plt.show()

              precision    recall  f1-score   support

     Healthy       0.82      0.82      0.82       274
     Disease       0.78      0.78      0.78       226

    accuracy                           0.80       500
   macro avg       0.80      0.80      0.80       500
weighted avg       0.80      0.80      0.80       500



C:\Users\DELL\AppData\Local\Temp\ipykernel_10104\2912960775.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6️⃣ تفسير العوامل الطبية (Feature Importance)

In [7]:
rf = Pipeline([('p',pre),('m',models['RandomForest'])]).fit(X_tr, y_tr)
names = rf.named_steps['p'].get_feature_names_out()
imp = pd.Series(rf.named_steps['m'].feature_importances_, index=names).sort_values().tail(10)
imp.plot(kind='barh', figsize=(7,4), title='Top risk factors'); plt.tight_layout(); plt.show()

C:\Users\DELL\AppData\Local\Temp\ipykernel_10104\1931696946.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  imp.plot(kind='barh', figsize=(7,4), title='Top risk factors'); plt.tight_layout(); plt.show()


## 7️⃣ الخلاصة والتوصيات (Conclusion)
- **درس التسريب:** شيلنا `risk_score` لأنها مشتقة من النتيجة — لو سِبناها كانت الدقة "وهمية".
- **النتيجة:** بعد إزالة التسريب، الموديل حقّق ROC-AUC واقعي وكويس بالعوامل الطبية الحقيقية.
- **أهم العوامل:** السن، الكوليسترول، ضغط الدم الانقباضي — متوافقة مع الطب المعروف ✓ (ثقة في الموديل).
- **التوصية:** استخدامه كأداة فرز مساعِدة توجّه الانتباه للحالات عالية الخطورة، مع تأكيد الطبيب دائماً.
- **الخطوة القادمة:** معايرة الاحتمالات + تحليل بـ SHAP لكل مريض على حدة.

> ✅ **اللي اتعلمته:** كشف الـ leakage من ميزة مشتقة، Pipeline، مقارنة موديلات، وتفسير طبي.
